# Notebook 08: Orchestration with Airflow

You have built all the pieces of a production ML system: data pipelines, feature engineering, model training, evaluation, serving, monitoring, CI/CD, and deployment. But who runs all of these? When? In what order? What happens when something fails?

That is the job of an **orchestrator**.

### What You Will Learn

- Why orchestration matters and what happens without it
- Apache Airflow fundamentals: DAGs, tasks, operators, schedules
- How our training, inference, and monitoring DAGs work
- How to run Airflow with Docker Compose
- How all three DAGs form a complete self-maintaining system

---
## 1. Why Orchestration?

Without an orchestrator, you are stuck with:

- **Cron jobs**: Schedule scripts with `crontab`. No dependency management, no retries, no visibility. If step 2 fails, step 3 runs anyway on stale data. When something breaks at 2 AM, you find out Monday morning.
- **Manual triggers**: Someone remembers to run the retraining script. Or they forget. Or they run it with the wrong parameters.
- **Ad-hoc scripts**: A chain of bash scripts calling Python scripts. No logging, no alerts, no way to see what ran when.

With an orchestrator:

- **Scheduled execution**: Pipelines run on time, every time.
- **Dependency management**: Step 3 only runs after step 2 succeeds.
- **Retries**: If a step fails due to a transient error, it retries automatically.
- **Monitoring and alerting**: A web UI shows what is running, what failed, and why.
- **Backfills**: Missed a day? Re-run just the missing days without re-running everything.

Think of the orchestrator as **the conductor of an orchestra**. Each musician (pipeline step) knows how to play their part. But without the conductor, they do not know when to start, when to stop, or how to stay in sync with each other.

---
## 2. What is Airflow?

**Apache Airflow** is the most widely used workflow orchestration platform in the data/ML world. It was created at Airbnb in 2014 and is now an Apache Software Foundation top-level project.

### Core Concepts

**DAG (Directed Acyclic Graph)**: A workflow defined as a graph of tasks with dependencies. "Directed" means tasks flow in one direction. "Acyclic" means no loops -- a task cannot depend on itself.

```
Example DAG: Training Pipeline

  generate_data
       |
       v
  validate_data
       |
       v
  engineer_features
       |
       v
  train_model
       |
       v
  evaluate_model
       |
       v
  register_model
```

**Task**: A single unit of work. "Generate data", "Train model", "Send alert" are all tasks.

**Operator**: The type of work a task does. Common operators:
- `PythonOperator` -- runs a Python function
- `BashOperator` -- runs a shell command
- `DockerOperator` -- runs a Docker container
- `EmailOperator` -- sends an email

**Schedule**: When the DAG runs. Examples:
- `@daily` -- once a day at midnight
- `@weekly` -- once a week on Sunday
- `@hourly` -- every hour
- `0 6 * * MON-FRI` -- 6 AM on weekdays (cron syntax)

**XCom (Cross-Communication)**: How tasks pass data to each other. Task A pushes a value, Task B pulls it. Example: the `generate_data` task pushes the file path, and `validate_data` pulls it to know where the data is.

**The Airflow Architecture:**

```
+-------------------+     +-------------------+
| Airflow Webserver |     | Airflow Scheduler |
| (UI at :8080)     |     | (triggers DAGs)   |
+--------+----------+     +--------+----------+
         |                         |
         +--------+   +-----------+
                  |   |
                  v   v
         +--------+---+---------+
         |   PostgreSQL DB      |
         | (DAG state, history) |
         +----------------------+
```

The **scheduler** checks every few seconds if any DAGs need to run. When a DAG is triggered, it executes tasks in dependency order. The **webserver** provides a UI to monitor, trigger, and debug DAGs.

---
## 3. Our Training DAG

The training DAG runs **weekly**. It orchestrates the full ML pipeline: generate data, validate it, engineer features, train a model, evaluate it, and register it in the model registry.

Let's read the actual code.

In [ ]:
# Read and display the training DAG
with open("../airflow/dags/training_dag.py") as f:
    print(f.read())

### Walking Through the Training DAG

**Default Arguments** (`default_args`):
- `owner: ml-engineering` -- who is responsible for this DAG
- `retries: 2` -- each task retries twice on failure
- `retry_delay: 10 minutes` -- waits 10 minutes between retries
- `execution_timeout: 2 hours` -- kills the task if it runs longer than 2 hours

**Task 1 -- `generate_data`**: Reads the data configuration and uses `DataGenerator` to fetch or generate training data. Pushes the data path to XCom so downstream tasks can find it.

**Task 2 -- `validate_data`**: Pulls the data path from XCom, runs `DataValidator`. If validation fails, the task raises a `ValueError` and the pipeline stops. This is a critical gate -- we never train on bad data.

**Task 3 -- `engineer_features`**: Applies the feature engineering pipeline. Reads model config to know which features to create. Pushes the features path.

**Task 4 -- `train_model`**: Trains the model and logs everything to MLflow. Pushes the MLflow `run_id` so we can find the model later.

**Task 5 -- `evaluate_model`**: Evaluates the trained model on holdout data. Pushes the metrics dictionary.

**Task 6 -- `register_model`**: Registers the model in MLflow Model Registry and transitions it to the "Staging" stage. From there, a human (or another automated process) can promote it to "Production".

**The chain**: `generate >> validate >> features >> train >> evaluate >> register`

Each `>>` means "the right side depends on the left side". If validation fails, nothing downstream runs.

---
## 4. Our Inference DAG

The inference DAG runs **daily**. It loads the production model, prepares new data, runs predictions, stores the results, and checks for drift.

In [ ]:
# Read and display the batch inference DAG
with open("../airflow/dags/batch_inference_dag.py") as f:
    print(f.read())

### Walking Through the Inference DAG

**Task 1 -- `load_model`**: Queries the MLflow Model Registry for the latest model in "Production" stage. Falls back to "Staging" if no Production model exists. Pushes the model URI and version.

**Task 2 -- `prepare_data`**: Generates or fetches a batch of input data for the current execution date. This is the data we want predictions for.

**Task 3 -- `run_predictions`**: Loads the model using `mlflow.pyfunc.load_model`, reads the batch data, runs `model.predict()`, and saves predictions to a Parquet file.

**Task 4 -- `store_results`**: Writes the predictions to the configured output storage (could be a database, S3, or a data warehouse).

**Task 5 -- `run_monitoring`**: Checks the predictions for drift. If the distribution of predictions looks different from training time, it logs a warning.

**The chain**: `load_model >> prepare_data >> run_predictions >> store_results >> run_monitoring`

This runs every day at midnight. By the time the team arrives in the morning, fresh predictions are ready.

---
## 5. Our Monitoring DAG

The monitoring DAG also runs **daily**. It computes a comprehensive drift report, checks against thresholds, and sends alerts if the model has degraded. This is the safety net that closes the loop.

In [ ]:
# Read and display the monitoring DAG
with open("../airflow/dags/monitoring_dag.py") as f:
    print(f.read())

### Walking Through the Monitoring DAG

**Task 1 -- `compute_drift_report`**: Uses `DriftDetector` to compute a full drift analysis comparing current data distributions to training data. Tracks which features have drifted and whether prediction distributions have shifted.

**Task 2 -- `check_thresholds`**: Compares the drift report against configurable thresholds:
- Feature drift: no more than 30% of features should drift
- Prediction drift score: should stay below 0.15
- Model performance: should stay above 0.85
If any threshold is violated, it generates alerts.

**Task 3 -- `alert_if_degraded`**: If thresholds were violated:
- Pushes metrics to Prometheus (for Grafana dashboards)
- Sends a webhook notification (Slack, Teams, PagerDuty)
- Logs warnings for the Airflow UI

**The chain**: `compute_drift_report >> check_thresholds >> alert_if_degraded`

This DAG is the **watchdog**. It catches model degradation before it causes real damage.

---
## 6. Running Airflow

To run Airflow locally, you use Docker Compose:

```bash
# Start all services (includes Airflow, PostgreSQL, MLflow, etc.)
docker compose up -d

# Or start just Airflow components
docker compose up -d airflow-init airflow-webserver airflow-scheduler postgres
```

Once running, open the Airflow UI at **http://localhost:8080**.

Default credentials (from the `airflow-init` container):
- Username: `admin`
- Password: `admin`

### Using the Airflow UI

1. **DAGs Page**: Shows all DAGs, their schedule, last run status, and next run time.
2. **Toggle ON/OFF**: DAGs are paused by default. Toggle them on to start scheduling.
3. **Trigger DAG**: Click the play button to run a DAG immediately (useful for testing).
4. **Graph View**: Visual representation of the DAG with task status colors.
5. **Task Logs**: Click any task to see its detailed execution logs.

```
Airflow UI Layout:

+------------------------------------------------------------+
| DAG                  | Schedule | Last Run | Status | Next  |
+------------------------------------------------------------+
| energy_demand_train  | @weekly  | Mar 30   |  OK    | Apr 6 |
| energy_demand_infer  | @daily   | Apr 6    |  OK    | Apr 7 |
| energy_demand_monitor| @daily   | Apr 6    |  WARN  | Apr 7 |
+------------------------------------------------------------+
```

### Useful Airflow CLI Commands

```bash
# List all DAGs
airflow dags list

# Trigger a DAG manually
airflow dags trigger energy_demand_training

# Check task status
airflow tasks state energy_demand_training train_model 2024-01-01

# View task logs
airflow tasks log energy_demand_training train_model 2024-01-01
```

---
## 7. The Complete System: How It All Works Together

The three DAGs form a self-maintaining ML system:

```
              +---------------------+
              |   TRAINING DAG      |
              |   (@weekly)         |
              |                     |
              | generate_data       |
              | validate_data       |
              | engineer_features   |
              | train_model         |
              | evaluate_model      |
              | register_model -----+---> MLflow Model Registry
              +---------------------+              |
                                                   |
                                                   v
              +---------------------+     (loads latest model)
              |   INFERENCE DAG     |              |
              |   (@daily)          |<-------------+
              |                     |
              | load_model          |
              | prepare_data        |
              | run_predictions     |
              | store_results       |
              | run_monitoring -----+---> Prediction Data Store
              +---------------------+              |
                                                   |
                                                   v
              +---------------------+     (analyzes predictions)
              |   MONITORING DAG    |              |
              |   (@daily)          |<-------------+
              |                     |
              | compute_drift       |
              | check_thresholds    |
              | alert_if_degraded --+---> Slack / PagerDuty / Grafana
              +---------------------+
                       |
                       | (if degradation detected)
                       v
              Team triggers retraining
              (or automated retrain trigger)
                       |
                       +---> Back to TRAINING DAG
```

### The Flow

1. **Every week**, the Training DAG generates data, validates it, engineers features, trains a new model, evaluates it, and registers it in MLflow. The model starts in "Staging".

2. **Every day**, the Inference DAG loads the latest Production model, prepares input data, runs batch predictions, and stores the results. Building operators use these predictions to optimize energy usage.

3. **Every day**, the Monitoring DAG analyzes recent predictions for drift and quality degradation. If thresholds are violated, alerts fire.

4. **If the model degrades**, the team is alerted and can trigger retraining. In a fully automated system (Level 3 MLOps), the monitoring DAG would trigger the training DAG automatically.

This is the **MLOps cycle** from Notebook 00, running on autopilot.

---
## 8. Exercises

### Exercise 1: Add a Branching Task to the Training DAG

Currently, the training DAG always registers the model. In practice, you would only register it if it is better than the current production model. Using Airflow's `BranchPythonOperator`, design a task that:

1. Pulls the evaluation metrics from XCom
2. Compares them to the current production model's metrics
3. If the new model is better, proceeds to `register_model`
4. If not, skips registration and logs a message

Hint: `BranchPythonOperator` returns the `task_id` of the next task to run.

### Exercise 2: Add a Notification Task to the Inference DAG

After the inference DAG completes, add a final task that sends a summary notification (e.g., to Slack or email) containing:
- Number of predictions made
- Model version used
- Whether any drift was detected

Write the Python callable and the `PythonOperator` definition.

---
## 9. Key Takeaways

1. **Orchestration is the glue that holds ML systems together.** Without it, you have scripts and cron jobs. With it, you have a reliable, observable, self-maintaining system.

2. **Airflow uses DAGs to define workflows.** Tasks are connected by dependencies, ensuring things run in the right order. If a step fails, downstream steps do not run.

3. **XCom passes data between tasks.** This keeps tasks loosely coupled -- each task does one thing and communicates through a standard mechanism.

4. **Three DAGs cover the full cycle.** Training (weekly) produces models. Inference (daily) produces predictions. Monitoring (daily) catches degradation. Together, they implement the MLOps lifecycle.

5. **The Airflow UI provides visibility.** You can see what ran, when it ran, whether it succeeded, and read the logs. No more guessing what happened at 2 AM.

6. **Retries and alerts handle failures gracefully.** Transient errors are retried automatically. Persistent failures trigger alerts.

In the next (and final) notebook, we will put everything together into one end-to-end demo and review everything you have learned across all nine notebooks.